# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook offers a complete walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, referencing all data entities (record sets, fields, columns) strictly via their `@id` values for reproducibility.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Install mlcroissant (if not already installed)
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and prepare to explore available entities.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
meta = dataset.metadata
print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Authors: {[author['@id'] if isinstance(author, dict) and '@id' in author else author for author in getattr(meta, 'author', [])]}")
print(f"Published: {getattr(meta, 'datePublished', 'n/a')}")

## 2. Data Overview

Review all available **record sets**, including their `@id`s, as well as associated field `@id`s. This is essential for referencing them when extracting and processing data.

In [ ]:
from pprint import pprint

print("Available record sets in schema:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}  |  Name: {rs.get('name', 'n/a')}")

# Show all fields within each record set by their @id
record_set_fields = {}
for rs in record_sets:
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nFields in record set '@id': {rs['@id']}:")
    field_ids = []
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field['@id']}")
            field_ids.append(field['@id'])
        elif isinstance(field, str):
            print(f"  - {field}")
            field_ids.append(field)
    record_set_fields[rs['@id']] = field_ids

# Display columns in the first available record set (if any)
first_rs = record_sets[0]['@id'] if record_sets else None
if first_rs:
    print(f"\nNow previewing first few records from record set '@id': {first_rs}")
    for i, rec in enumerate(dataset.records(record_set=first_rs)):
        if i < 2:
            pprint(rec)
        else:
            break
else:
    print("No record sets detected in this schema.")

## 3. Data Extraction

Extract each record set's data into Pandas DataFrames for analysis.

> **Tip:** Reference entities (record sets and fields) always by their `@id` for full schema traceability.

In [ ]:
# List of record set @ids (obtain from above code, or from the schema itself)
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Extracting DataFrames for record sets: ", all_record_set_ids)

dataframes = dict()
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns in each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns in DataFrame for record set '@id': {rs_id}")
    print(list(df.columns))
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Typical preprocessing tasks:
- Filter numeric or categorical fields (e.g., remove rows with missing or extreme values by a field `@id`)
- Normalize/standardize numeric columns using their `@id`
- Group by or aggregate over a specific field

**Note:**
- All references to fields/columns use their Croissant `@id`.
- You may need to adapt the IDs or preprocessing steps once you inspect the dataframe columns above.

In [ ]:
# Example: EDA on a selected record set (replace these @ids with real ones from step 2)
# If you see no record sets in step 2, leave this cell as is for demonstration.

if dataframes:
    # Pick the first available record set and numeric field (as examples)
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]

    # Find a likely numeric field by inspecting dtypes or known schema ids
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric field candidates for record set '@id': {selected_rs_id}: {numeric_candidates}")
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

        # Filter and normalize
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records in '{selected_rs_id}' with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a likely categorical/group field (pick first object dtype not the same as numeric field)
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo categorical/group field found for groupby demonstration.")
    else:
        print("No numeric field detected in first record set for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize the distribution of a numeric variable or the relationship between fields, referencing all entities by their `@id`.

> For demonstration, a histogram and a boxplot for the selected numeric field are produced. Adapt field `@id`s as needed for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example using the same numeric_field_id as in EDA
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Histogram: {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and explore a Croissant-described dataset using `mlcroissant`
- Enumerate and reference all data structures (record sets, fields) by their persistent `@id`
- Extract tabular data for further analysis
- Apply basic filtering, statistics, and visualizations using standard Python libraries

This approach ensures transparent and reproducible data referencing. For in-depth analysis, adapt grouping, filtering, or modeling logic using the discovered Croissant `@id`s for your specific research question.
